In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.feature_selection import SelectKBest, mutual_info_classif
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report
from imblearn.over_sampling import SMOTE
import warnings

warnings.filterwarnings('ignore')

df = pd.read_csv('liver_cirrhosis.csv')
target_col = 'Stage'
if target_col not in df.columns:
    raise ValueError(f"Target column '{target_col}' does not exist!")
df = df.dropna(subset=[target_col])
leakage_features = [
    'N_Days',
    'Status',
]
removed_leakage = [f for f in leakage_features if f in df.columns]
df = df.drop(columns=removed_leakage)

# Delete features with high missing rate
missing_rates = df.isnull().sum() / len(df)
high_missing = missing_rates[missing_rates > 0.8].index.tolist()
high_missing = [c for c in high_missing if c != target_col]
if high_missing:
    df = df.drop(columns=high_missing)
    print(f"{len(high_missing)} features with high missing rate have been removed")

exclude_kw = ['stage', 'unnamed', 'barcode', 'uuid', 'patient_id',
              'bcr_', 'icd_', 'project', 'has_', 'vital', 'days_to', 'diagnosis']

feature_cols = []
for col in df.columns:
    if col == target_col:
        continue
    if any(kw in col.lower() for kw in exclude_kw):
        continue
    if df[col].isnull().sum() / len(df) < 0.5:
        feature_cols.append(col)

print(f"{len(feature_cols)} candidate features have been identified")

X = df[feature_cols].copy()
y = df[target_col].copy()

# Handle missing values
for col in X.select_dtypes(include=[np.number]).columns:
    if X[col].isnull().any():
        X[col].fillna(X[col].median(), inplace=True)

for col in X.select_dtypes(include=['object']).columns:
    if X[col].isnull().any():
        mode = X[col].mode()[0] if len(X[col].mode()) > 0 else 'Unknown'
        X[col].fillna(mode, inplace=True)

# Label encoding
for col in X.select_dtypes(include=['object']).columns:
    le = LabelEncoder()
    X[col] = le.fit_transform(X[col].astype(str))

print(f"Feature matrix: {X.shape}")

# Feature selection - intelligently select optimal number of features
best_k = 10
best_score = 0

print("Testing different numbers of features...")
for k in [5, 10, 15, 20, 25, 30]:
    if k > len(feature_cols):
        break

    selector_tmp = SelectKBest(score_func=mutual_info_classif, k=k)
    X_k = selector_tmp.fit_transform(X, y)

    rf_temp = RandomForestClassifier(n_estimators=50, random_state=42, max_depth=5)
    scores = cross_val_score(rf_temp, X_k, y, cv=3, scoring='accuracy')
    avg_score = scores.mean()

    print(f"  K={k:2d}: CV Accuracy = {avg_score:.4f} (±{scores.std():.4f})")

    if avg_score > best_score:
        best_score = avg_score
        best_k = k

print(f"\nSelected K={best_k} features, expected CV accuracy={best_score:.4f}")

# Use best K value to select features
selector = SelectKBest(
    score_func=lambda X, y: mutual_info_classif(X, y, random_state=42),
    k=best_k
)
selector.fit(X, y)

feature_scores = pd.DataFrame({
    'feature': feature_cols,
    'score': selector.scores_
}).sort_values('score', ascending=False)

print(feature_scores.head(20).to_string(index=False))

selected_mask = selector.get_support()
selected_features = [feature_cols[i] for i in range(len(feature_cols)) if selected_mask[i]]

for i, feat in enumerate(selected_features, 1):
    score = feature_scores[feature_scores['feature'] == feat]['score'].values[0]
    print(f"  {i:2d}. {feat:<65} (score: {score:.4f})")

X_selected = X[selected_features].copy()

# Data splitting
class_counts = y.value_counts()
print("\nSample count per stage:")
for stage, count in class_counts.items():
    print(f"  {stage}: {count} cases")

min_samples = class_counts.min()
stratify_param = None if min_samples < 2 else y

X_train, X_test, y_train, y_test = train_test_split(
    X_selected, y,
    test_size=0.3,
    random_state=42,
    stratify=stratify_param
)

print(f"\nTraining set: {X_train.shape}")
print(f"Test set: {X_test.shape}")

# Standardization (before SMOTE)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

# SMOTE
smote = SMOTE(random_state=42)
X_train_bal, y_train_bal = smote.fit_resample(X_train_scaled, y_train)

print(f"\nBefore SMOTE: {X_train_scaled.shape}")
print(f"After SMOTE: {X_train_bal.shape}")
print(f"Added samples: {X_train_bal.shape[0] - X_train_scaled.shape[0]}")

print("\nTraining set distribution (after SMOTE):")
for s, c in pd.Series(y_train_bal).value_counts().sort_index().items():
    print(f"  {s}: {c} cases")

# GBM
gbm_model = GradientBoostingClassifier(
    n_estimators=200,
    learning_rate=0.05,
    max_depth=3,
    min_samples_split=2,
    min_samples_leaf=1,
    subsample=1.0,
    max_features='sqrt',
    random_state=42,
    verbose=0
)

gbm_model.fit(X_train_bal, y_train_bal)

# Cross-validation
X_selected_scaled = scaler.transform(X_selected)
cv_scores = cross_val_score(gbm_model, X_selected_scaled, y, cv=5, scoring='accuracy')
for i, score in enumerate(cv_scores, 1):
    print(f"  Fold {i}: {score:.4f}")
print(f"\naverage: {cv_scores.mean():.4f} (±{cv_scores.std():.4f})")

# Importance of selected features
final_importance = pd.DataFrame({
    'feature': selected_features,
    'importance': gbm_model.feature_importances_
}).sort_values('importance', ascending=False)

print(f"\nModel importance of these {len(selected_features)} features:")
print(final_importance.to_string(index=False))

# Model evaluation
y_train_pred = gbm_model.predict(X_train_scaled)

train_accuracy  = accuracy_score(y_train, y_train_pred)
train_precision = precision_score(y_train, y_train_pred, average='weighted', zero_division=0)
train_recall    = recall_score(y_train, y_train_pred, average='weighted', zero_division=0)
train_f1        = f1_score(y_train, y_train_pred, average='weighted', zero_division=0)

y_test_pred = gbm_model.predict(X_test_scaled)

test_accuracy  = accuracy_score(y_test, y_test_pred)
test_precision = precision_score(y_test, y_test_pred, average='weighted', zero_division=0)
test_recall    = recall_score(y_test, y_test_pred, average='weighted', zero_division=0)
test_f1        = f1_score(y_test, y_test_pred, average='weighted', zero_division=0)

print("=" * 70)
print(" " * 20 + "Model Performance Comparison")
print("=" * 70)
print(f"{'Metric':<15} {'Training Set':>15} {'Test Set':>15} {'Difference':>15}")
print("-" * 70)
print(f"{'Accuracy':<15} {train_accuracy:>15.4f} {test_accuracy:>15.4f} {abs(train_accuracy-test_accuracy):>15.4f}")
print(f"{'Precision':<15} {train_precision:>15.4f} {test_precision:>15.4f} {abs(train_precision-test_precision):>15.4f}")
print(f"{'Recall':<15} {train_recall:>15.4f} {test_recall:>15.4f} {abs(train_recall-test_recall):>15.4f}")
print(f"{'F1 Score':<15} {train_f1:>15.4f} {test_f1:>15.4f} {abs(train_f1-test_f1):>15.4f}")
print("=" * 70)

print("\nClassification Report:")
print(classification_report(y_test, y_test_pred, zero_division=0))